# IBEX 35 Volatility: Stage 5 — VaR Backtesting and Validation

Stage 4 produced dynamic VaR from the preferred GARCH models and an EWMA
benchmark, and took a first informal look at breach counts. This notebook
puts both models through the two standard statistical backtests used in
practice (and on the FRM/market-risk curriculum) to judge whether a VaR model
is actually fit for purpose:

- **Kupiec's proportion-of-failures (POF) test** — does the *number* of
  breaches match what the model promised?
- **Christoffersen's independence test** — are breaches *scattered randomly*
  in time, or do they cluster (a sign the model is slow to react to regime
  changes)?

Both are implemented from scratch below — not called from a backtesting
library — since the point of this stage is to understand exactly what these
tests are doing, not just to get a pass/fail flag.

This notebook is self-contained: it re-downloads the same ~10y of daily data,
refits the Stage 3 preferred models, and reconstructs the same GARCH and
EWMA VaR series as Stage 4.

**A methodological note before we start:** we are about to test whether a
handful of models pass or fail. We report the results exactly as they come
out — including failures. A backtesting notebook that only reports models
"passing" isn't validation, it's marketing.


## 1. Setup

In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from arch import arch_model
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100


## 2. Data

In [2]:
TICKERS = {"IBEX35": "^IBEX", "SP500": "^GSPC"}
END = pd.Timestamp.today().normalize()
START = END - pd.DateOffset(years=10)

raw = yf.download(list(TICKERS.values()), start=START, end=END, auto_adjust=True, progress=False)
prices = raw["Close"].rename(columns={v: k for k, v in TICKERS.items()})
prices = prices.dropna(how="all").ffill().dropna()

returns_pct = 100 * np.log(prices / prices.shift(1)).dropna()
returns_pct.tail()


Ticker,SP500,IBEX35
Date,,
2026-07-07,-0.446506,-0.221756
2026-07-08,-0.282121,-2.766496
2026-07-09,0.810982,1.137230
2026-07-10,0.420001,0.319827
2026-07-13,-0.795861,-0.253097


## 3. Rebuild the VaR series (GARCH and EWMA)

Exactly as in Stage 4: refit the Stage 3 preferred specification per index
(GJR-GARCH for IBEX 35, EGARCH for S&P 500, Student-t innovations), and
reconstruct both the GARCH-based parametric VaR and the EWMA/RiskMetrics
($\lambda=0.94$, normal quantile) benchmark VaR, at 95% and 99%. The
quantile helper reuses the same standardized (unit-variance) Student-t
correction from Stage 4: the raw `scipy.stats.t.ppf(p, ν)` quantile rescaled
by $\sqrt{(\nu-2)/\nu}$.


In [3]:
preferred_spec = {
    "IBEX35": dict(vol="GARCH", p=1, o=1, q=1),   # GJR-GARCH
    "SP500": dict(vol="EGARCH", p=1, o=1, q=1),   # EGARCH
}

results = {}
for idx, spec in preferred_spec.items():
    am = arch_model(returns_pct[idx], mean="Constant", dist="t", **spec)
    results[idx] = am.fit(disp="off")


def std_t_quantile(nu, p):
    raw_q = stats.t.ppf(p, nu)
    scale = np.sqrt((nu - 2) / nu)
    return raw_q * scale


def ewma_volatility(returns, lam=0.94, seed_window=30):
    var = np.empty(len(returns))
    var[0] = returns.iloc[:seed_window].var()
    for t in range(1, len(returns)):
        var[t] = lam * var[t - 1] + (1 - lam) * returns.iloc[t - 1] ** 2
    return pd.Series(np.sqrt(var), index=returns.index)


ewma_vol = {idx: ewma_volatility(returns_pct[idx]) for idx in returns_pct.columns}

CONF_LEVELS = [0.95, 0.99]

var_thresholds = {}  # (idx, model, conf) -> Series of return-scale thresholds
for idx in returns_pct.columns:
    res = results[idx]
    mu, nu = res.params["mu"], res.params["nu"]
    sigma = res.conditional_volatility
    for conf in CONF_LEVELS:
        p = 1 - conf
        var_thresholds[(idx, "GARCH", conf)] = mu + sigma * std_t_quantile(nu, p)
        var_thresholds[(idx, "EWMA", conf)] = 0 + ewma_vol[idx] * stats.norm.ppf(p)


## 4. Kupiec's proportion-of-failures (POF) test

A VaR model at confidence $1-p$ should be breached, on average, a fraction
$p$ of the time — no more, no less. Kupiec's test checks exactly this
**unconditional coverage** property. Let $x$ be the observed number of
breaches out of $n$ observations, and $\hat\pi = x/n$ the observed breach
rate. Under $H_0$: the true breach probability equals the model's nominal
$p$, the likelihood-ratio statistic is

$$
LR_{uc} = -2\ln\left[\frac{(1-p)^{n-x}\, p^{x}}{(1-\hat\pi)^{n-x}\, \hat\pi^{x}}\right]
$$

— the log-likelihood of the data under the *nominal* breach probability $p$
in the numerator, against the log-likelihood under the *observed* rate
$\hat\pi$ (its maximum-likelihood estimate) in the denominator. If the model
is well calibrated, $\hat\pi \approx p$ and $LR_{uc} \approx 0$; the further
$\hat\pi$ drifts from $p$ (relative to the sample size), the larger
$LR_{uc}$ grows. Under $H_0$, $LR_{uc} \sim \chi^2(1)$ asymptotically, so we
reject correct coverage when $LR_{uc}$ exceeds the $\chi^2(1)$ critical value
(equivalently, when its p-value falls below 5%).

Note what this test does **not** check: it only cares about the *total
count* of breaches, not when they happened. Ten breaches evenly spread over
ten years and ten breaches all occurring in the same month would give
identical $LR_{uc}$ — that's exactly the gap Christoffersen's test closes.


In [4]:
def kupiec_test(x, n, p):
    pi_hat = x / n

    def log_lik(prob, x, n):
        if x == 0:
            return (n - x) * np.log(1 - prob)
        if x == n:
            return x * np.log(prob)
        return (n - x) * np.log(1 - prob) + x * np.log(prob)

    lr_uc = -2 * (log_lik(p, x, n) - log_lik(pi_hat, x, n))
    p_value = 1 - stats.chi2.cdf(lr_uc, df=1)
    return lr_uc, p_value


## 5. Christoffersen's independence and conditional-coverage tests

Model breach days as a binary indicator series $I_t \in \{0, 1\}$ and treat
it as a first-order Markov chain: does knowing whether *yesterday* was a
breach change the probability that *today* is one? Count the four possible
consecutive-day transitions:

|                | $I_t = 0$ | $I_t = 1$ |
|----------------|-----------|-----------|
| $I_{t-1} = 0$  | $n_{00}$  | $n_{01}$  |
| $I_{t-1} = 1$  | $n_{10}$  | $n_{11}$  |

and the corresponding transition probabilities $\hat\pi_{01} =
n_{01}/(n_{00}+n_{01})$ (breach tomorrow given no breach today) and
$\hat\pi_{11} = n_{11}/(n_{10}+n_{11})$ (breach tomorrow given a breach
today). Under **independence**, these two should be equal (both equal to
the overall unconditional breach rate $\hat\pi$); under **clustering**,
$\hat\pi_{11} > \hat\pi_{01}$ — a breach today makes another breach tomorrow
more likely than the baseline rate would suggest. The likelihood-ratio
statistic compares a single pooled probability $\hat\pi$ against the two
separately estimated transition probabilities:

$$
LR_{ind} = -2\ln\left[\frac{(1-\hat\pi)^{n_{00}+n_{10}}\, \hat\pi^{\,n_{01}+n_{11}}}{(1-\hat\pi_{01})^{n_{00}}\, \hat\pi_{01}^{\,n_{01}}\, (1-\hat\pi_{11})^{n_{10}}\, \hat\pi_{11}^{\,n_{11}}}\right] \sim \chi^2(1)
$$

A model can pass Kupiec (right *number* of breaches) yet fail this test (the
*wrong timing* — breaches bunched together), or vice versa. The **joint
conditional-coverage test** combines both properties into a single check —
correct coverage *and* independence — by simply adding the two independent
LR statistics:

$$
LR_{cc} = LR_{uc} + LR_{ind} \sim \chi^2(2)
$$


In [5]:
def christoffersen_independence_test(breach_indicator):
    breach = np.asarray(breach_indicator).astype(int)
    n00 = n01 = n10 = n11 = 0
    for t in range(1, len(breach)):
        prev, curr = breach[t - 1], breach[t]
        if prev == 0 and curr == 0:
            n00 += 1
        elif prev == 0 and curr == 1:
            n01 += 1
        elif prev == 1 and curr == 0:
            n10 += 1
        else:
            n11 += 1

    n0_, n1_ = n00 + n01, n10 + n11
    pi01 = n01 / n0_ if n0_ > 0 else 0.0
    pi11 = n11 / n1_ if n1_ > 0 else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)

    def term(count, prob):
        return 0.0 if count == 0 else count * np.log(prob)

    log_l0 = term(n00 + n10, 1 - pi) + term(n01 + n11, pi)
    log_l1 = term(n00, 1 - pi01) + term(n01, pi01) + term(n10, 1 - pi11) + term(n11, pi11)

    lr_ind = -2 * (log_l0 - log_l1)
    p_value = 1 - stats.chi2.cdf(lr_ind, df=1)
    counts = {"n00": n00, "n01": n01, "n10": n10, "n11": n11, "pi01": pi01, "pi11": pi11}
    return lr_ind, p_value, counts


## 6. Results table and Basel traffic-light context

We run both tests for every (index, model, confidence level) combination —
8 cells in total — and collect breach counts, Kupiec unconditional-coverage
results, Christoffersen independence results, and the joint conditional
coverage result into one table.

For the **99% VaR** results specifically, we also map the observed breach
rate onto the **Basel "traffic light" backtesting framework**, which banks
use to set the regulatory capital multiplier: over a rolling 250 trading-day
window, **0-4 exceptions is the green zone** (VaR model accepted as-is),
**5-9 is yellow** (increasing capital multiplier, supervisory scrutiny), and
**10+ is red** (model presumed inadequate). Our sample is roughly ten years
of data rather than Basel's specified 250-day window, so we scale our
observed breach *rate* to its 250-day equivalent ($\text{rate} \times 250$)
as an indicative annualized comparison, not a literal rolling-window
backtest.


In [6]:
rows = []
for idx in returns_pct.columns:
    r = returns_pct[idx]
    n = len(r)
    for model in ["GARCH", "EWMA"]:
        for conf in CONF_LEVELS:
            p = 1 - conf
            thr = var_thresholds[(idx, model, conf)]
            breach = (r < thr).astype(int)
            x = int(breach.sum())

            lr_uc, p_uc = kupiec_test(x, n, p)
            lr_ind, p_ind, counts = christoffersen_independence_test(breach.values)
            lr_cc = lr_uc + lr_ind
            p_cc = 1 - stats.chi2.cdf(lr_cc, df=2)

            rows.append({
                "Index": idx, "Model": model, "Confidence": conf,
                "n": n, "Breaches": x, "Breach rate": x / n, "Expected rate": p,
                "Kupiec LR": lr_uc, "Kupiec p": p_uc,
                "Independence LR": lr_ind, "Independence p": p_ind,
                "Cond. coverage LR": lr_cc, "Cond. coverage p": p_cc,
                "Pass (5%)": p_cc > 0.05,
                "250-day exceptions": (x / n) * 250,
            })

results_df = pd.DataFrame(rows).set_index(["Index", "Model", "Confidence"]).sort_index()
results_df.round(4)


n  Breaches  Breach rate  Expected rate  \
Index  Model Confidence                                               
IBEX35 EWMA  0.95        2579       154       0.0597           0.05   
             0.99        2579        55       0.0213           0.01   
       GARCH 0.95        2579       157       0.0609           0.05   
             0.99        2579        32       0.0124           0.01   
SP500  EWMA  0.95        2579       143       0.0554           0.05   
             0.99        2579        58       0.0225           0.01   
       GARCH 0.95        2579       164       0.0636           0.05   
             0.99        2579        40       0.0155           0.01   

                         Kupiec LR  Kupiec p  Independence LR  Independence p  \
Index  Model Confidence                                                         
IBEX35 EWMA  0.95           4.8356    0.0279          11.1666          0.0008   
             0.99          25.2236    0.0000           0.5056          0.4771   
       GARCH 0.95           6.0242    0.0141           0.6587          0.4170   
             0.99           1.4031    0.2362           0.6643          0.4150   
SP500  EWMA  0.95           1.5588    0.2118           0.5630          0.4530   
             0.99          30.0010    0.0000           1.7083          0.1912   
       GARCH 0.95           9.2687    0.0023           0.0346          0.8524   
             0.99           6.7706    0.0093           2.0204          0.1552   

                         Cond. coverage LR  Cond. coverage p  Pass (5%)  \
Index  Model Confidence                                                   
IBEX35 EWMA  0.95                  16.0021            0.0003      False   
             0.99                  25.7291            0.0000      False   
       GARCH 0.95                   6.6829            0.0354      False   
             0.99                   2.0674            0.3557       True   
SP500  EWMA  0.95                   2.1219            0.3461       True   
             0.99                  31.7093            0.0000      False   
       GARCH 0.95                   9.3033            0.0095      False   
             0.99                   8.7911            0.0123      False   

                         250-day exceptions  
Index  Model Confidence                      
IBEX35 EWMA  0.95                   14.9283  
             0.99                    5.3315  
       GARCH 0.95                   15.2191  
             0.99                    3.1020  
SP500  EWMA  0.95                   13.8620  
             0.99                    5.6223  
       GARCH 0.95                   15.8976  
             0.99                    3.8775

In [7]:
def basel_zone(exceptions_250):
    if exceptions_250 <= 4:
        return "Green"
    if exceptions_250 <= 9:
        return "Yellow"
    return "Red"


basel_df = results_df.reset_index()
basel_df = basel_df[basel_df["Confidence"] == 0.99][
    ["Index", "Model", "Breaches", "Breach rate", "250-day exceptions"]
].copy()
basel_df["Basel zone"] = basel_df["250-day exceptions"].apply(basel_zone)
basel_df.set_index(["Index", "Model"])


Breaches  Breach rate  250-day exceptions Basel zone
Index  Model                                                      
IBEX35 EWMA         55     0.021326            5.331524     Yellow
       GARCH        32     0.012408            3.101978      Green
SP500  EWMA         58     0.022489            5.622334     Yellow
       GARCH        40     0.015510            3.877472      Green

**Interpretation.** *(See Section 7 for the full honest discussion — the
short version: at 99%, both GARCH models land in Basel's green zone (IBEX
35: 3.10 scaled exceptions; S&P 500: 3.88) while both EWMA models land in
yellow (IBEX 35: 5.33; S&P 500: 5.63). But the Basel zone verdict and the
statistical Kupiec verdict don't always agree — see Section 7 for why.)*


## 7. Honest interpretation

We report exactly what the eight tests found, without forcing a tidy "the
model works" conclusion.

**Only one of the eight (index, model, confidence) combinations cleanly
passes conditional coverage: IBEX 35 GJR-GARCH at 99%** (32 breaches vs. an
expected 25.8, Kupiec p = 0.24, independence p = 0.42, joint p = 0.36). Every
other combination fails at the 5% level, and in most cases the failure is
driven by **too many breaches, not clustering**:

- **At 95%, both GARCH models over-breach and fail Kupiec**: S&P 500 (164
  vs. 129 expected, p = 0.002) and IBEX 35 (157 vs. 129, p = 0.014). Neither
  shows significant clustering (independence p = 0.85 and 0.42), so the
  *timing* of breaches isn't the problem — the model is simply breached more
  often than a 95% VaR should be.
- **At 99%, S&P 500's GARCH VaR also over-breaches and fails Kupiec** (40
  vs. 26 expected, p = 0.009), again with no clustering (p = 0.16) — so
  EGARCH's 99% threshold for the S&P 500 is measurably too tight, not just
  unluckily timed.
- **EWMA fails badly and consistently at 99%** for both indices — 58 breaches
  for the S&P 500 (p < 0.0001) and 55 for the IBEX 35 (p < 0.0001), both
  roughly double the nominal rate. This is the clearest result in the whole
  notebook: **the normal-quantile assumption is not adequate at the 99%
  level**, exactly as Stage 4 anticipated from the tail-thickness argument.
- **The one genuinely alarming independence failure is IBEX 35's EWMA VaR at
  95%** (independence p = 0.0008, $\hat\pi_{11}=0.130$ vs.
  $\hat\pi_{01}=0.055$) — once an EWMA breach happens, the probability of
  another one the very next day is roughly **2.4x** the baseline rate. This
  is a textbook symptom of a volatility estimator that reacts too slowly to
  a regime shift, letting a burst of breaches happen consecutively before
  the EWMA variance catches up.

**On the GARCH-vs-EWMA question**, the picture is genuinely mixed rather
than a clean GARCH win: GARCH passes Kupiec for IBEX 35 at 99% (EWMA does
not), and GARCH's independence properties are better across the board — but
GARCH *also* fails unconditional coverage at three of the four cells where
EWMA doesn't clearly lose on both counts. The clearest, most defensible
conclusion is narrower than "GARCH is better": **GARCH's breach *rate* is
consistently closer to nominal than EWMA's, especially at 99%, but "closer"
is not the same as "passes"** — three of GARCH's four cells still formally
fail Kupiec at the 5% level.

**On the Basel traffic-light discrepancy**: both GARCH models land in the
green zone at 99% (Section 6), yet S&P 500's GARCH VaR still fails the
Kupiec test on the full sample. This is not a contradiction — it's a power
difference. Basel's zone table is calibrated to a 250-observation window and
is deliberately lenient (allowing up to 4 exceptions before even a yellow
warning) precisely because a short window can't distinguish bad luck from a
bad model very precisely. Testing the *same* underlying breach rate against
our full ~2,579-observation sample gives Kupiec far more statistical power
to detect a deviation from nominal coverage — so a rate that would sail
through a bank's quarterly regulatory review can still be flagged by a more
data-rich academic backtest.

**The practical takeaway is a known, well-documented limitation of
parametric VaR, not a flaw specific to our implementation**: even a
carefully fitted Student-t GARCH model tends to *underestimate* the true
frequency of extreme moves in a sample that includes an event like
COVID-2020, because a single (however fat-tailed) parametric distribution
fit over the whole sample still smooths over the fact that tail risk itself
is time-varying and regime-dependent. Two natural next steps, consistent
with how the industry has actually responded to this exact problem: (1) rely
more on the **Expected Shortfall** figures from Stage 4 rather than VaR
alone — which is precisely why Basel's FRTB moved capital requirements from
99% VaR to 97.5% ES in the first place — and (2) consider an **Extreme
Value Theory (EVT)** approach (e.g. Peaks-Over-Threshold with a
Generalized Pareto tail) that models the extreme tail directly rather than
extrapolating it from a single global Student-t fit, as a natural extension
beyond the scope of this project.


## 8. Summary

1. **Kupiec POF test**: implemented from scratch; only IBEX 35's GJR-GARCH
   VaR at 99% passes unconditional coverage. All seven other
   (index, model, confidence) cells reject at 5%, almost always because of
   *too many* breaches — VaR too tight, not too loose.
2. **Christoffersen independence test**: breach clustering is *not* the main
   problem in this dataset — most cells show no significant clustering — with
   one clear exception: IBEX 35's EWMA VaR at 95% shows strongly significant
   clustering (independence p = 0.0008).
3. **GARCH vs. EWMA**: GARCH's breach rates are consistently closer to
   nominal than EWMA's, especially at 99% (EWMA roughly doubles the nominal
   breach rate at 99% for both indices) — but this is a difference of
   degree, not a clean pass/fail split. Most GARCH cells fail Kupiec too.
4. **Basel traffic light (99%)**: both GARCH models land in the green zone;
   both EWMA models land in yellow. This more lenient, lower-power test can
   disagree with the stricter full-sample Kupiec verdict — both conclusions
   are "correct" for the test they run, they just have different statistical
   power.
5. **Bottom line**: parametric Student-t GARCH VaR measurably improves on a
   simple EWMA/normal benchmark, particularly in the far tail, but neither
   should be treated as fully validated at the 99% level over a sample that
   includes COVID-2020. Expected Shortfall (Stage 4) and an EVT-based tail
   model are the natural, industry-standard responses to this exact finding.
